<a href="https://colab.research.google.com/github/SarahkhIT/AgentsEngineeringProject/blob/main/notebooks/02_graph_orchestration_and_hitl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graph-Based Orchestration & Human-in-the-Loop
**Solar Farm Agentic System: Part 2 of 5**

Covers **Rubric Deliverable 2 (Graph-Based Orchestration)** and the
persistence/HITL half of **Deliverable 5**.

> **Run order:** this notebook continues from
> `01_agentic_reasoning_and_tools.ipynb` — run that notebook first in the
> same kernel session (or Kernel → Restart & Run All across `01` then `02`)
> so `SolarState`, the tool functions, and the node functions it defines are
> already in memory.

Builds the real `StateGraph`: nodes, a fixed edge chain, and two
**conditional edges** — `route_after_panel` (branches to a maintenance node
only when a panel group is actually faulty) and `route_after_energy` (the
**retry loop** — routes back to `increment_retry` while forecast confidence
is below 0.6, up to 3 attempts, before continuing). Adds a `SqliteSaver`
checkpointer so state survives a restart, and a real
`human_approval_node` that calls `interrupt()` and resumes on
`Command(resume=True)`.

The two cells at the end prove the retry loop actually fires and terminates
inside a live compiled graph, not just in a unit test of the routing
function.


In [ ]:
def aggregator_node(state: SolarState) -> SolarState:
    state["final_report"] = (
        f"Weather: {state['weather_data']}\n"
        f"Panels: {state['panel_status']}\n"
        f"Forecast: {state['energy_forecast']}\n"
        f"Maintenance needed: {state['maintenance_needed']}"
    )
    print("[Aggregator] Report compiled")
    return state

In [ ]:
def review_node(state: SolarState) -> SolarState:
    print("[Review Agent] Critiquing final report...")
    # Stub critique logic — swap for an LLM call later:
    # prompt = f"Review this solar farm report for gaps or inconsistencies:\n{state['final_report']}"
    # response = llm.invoke([HumanMessage(content=prompt)])
    issues = []
    if state["maintenance_needed"] and state["energy_forecast"]["confidence"] < 0.5:
        issues.append("Low-confidence forecast alongside a maintenance flag — recommend re-verification before acting.")
    state["review_notes"] = "; ".join(issues) if issues else "No issues found — report approved."
    print(f"[Review Agent] {state['review_notes']}")
    return state

In [ ]:
from langgraph.types import interrupt, Command

def human_approval_node(state: SolarState):
    approval = interrupt(
        {
            "message": "Do you approve this report?",
            "report": state["final_report"]
        }
    )

    state["human_approved"] = approval

    if approval:
        state["approval_message"] = "Report approved by human."
    else:
        state["approval_message"] = "Report rejected by human."

    return state

In [ ]:
def route_after_panel(state: SolarState) -> str:
    if any(v.get("fault") for v in state["panel_status"].values()):
        return "maintenance"
    return "aggregate"

def route_after_energy(state: SolarState) -> str:
    confidence = state["energy_forecast"].get("confidence", 0)
    retries = state.get("retries", 0)

    if confidence < 0.6 and retries < 3:
        return "retry"

    return "continue"

In [ ]:
def increment_retry_node(state: SolarState) -> SolarState:
    current_retries = state.get("retries", 0)
    new_retries = current_retries + 1

    print(
        f"[Retry Node] Confidence too low "
        f"({state['energy_forecast'].get('confidence', 0):.2f}) "
        f"— retrying. Attempt {new_retries}"
    )

    return {
        **state,
        "retries": new_retries
    }

In [ ]:
from langgraph.graph import StateGraph, END

graph = StateGraph(SolarState)
graph.add_node("planner", planner_node)
graph.add_node("weather", weather_node)
graph.add_node("panel", panel_node)
graph.add_node("energy", energy_node)
graph.add_node("increment_retry", increment_retry_node)
graph.add_node("maintenance", maintenance_node)
graph.add_node("aggregate", aggregator_node)
graph.add_node("human_approval", human_approval_node)
graph.set_entry_point("planner")
graph.add_edge("planner", "weather")
graph.add_edge("weather", "panel")

graph.add_conditional_edges("panel", route_after_panel, {
    "maintenance": "maintenance",
    "aggregate": "energy"
})
graph.add_edge("maintenance", "energy")

graph.add_conditional_edges("energy", route_after_energy, {
    "retry": "increment_retry",
    "continue": "aggregate"
})

graph.add_edge("increment_retry", "energy")
graph.add_node("review", review_node)
graph.add_edge("aggregate", "review")
graph.add_edge("review", "human_approval")
graph.add_edge("human_approval", END)
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

checkpoint_conn = sqlite3.connect(
    "checkpoints.sqlite",
    check_same_thread=False
)

checkpointer = SqliteSaver(checkpoint_conn)

app = graph.compile(checkpointer=checkpointer)



In [ ]:
config = {
    "configurable": {
        "thread_id": "solar-farm-hitl-run-1"
    }
}

result = app.invoke(
    {
        "task": "Assess solar farm status",
        "plan": [],
        "retries": 0
    },
    config=config
)

print(result["final_report"])

[Planner] Plan created: ['check_weather', 'analyze_panels', 'predict_energy', 'decide_maintenance']
[Weather Agent] Thought: I need the farm's current weather and solar irradiance before anything else.
[Weather Agent] Action: call get_weather(lat=..., lon=...)
[Weather Agent] Observation: {'source': 'open-meteo', 'condition': 'cloudy', 'irradiance': 0.0, 'cloud_cover_pct': 88, 'temperature_c': 42.1}
[Panel Analysis Agent] Thought: I need to check each panel group's sensor telemetry for underperformance.
[Panel Analysis Agent] Action: call detect_faults(num_groups=..., fault_threshold_pct=..., seed=..., force_fault_group=...)
[Panel Analysis Agent] Observation: {'groups': {'group_1': {'expected_kw': 10.0, 'actual_kw': 9.82, 'output_pct': 98.2, 'fault': False}, 'group_2': {'expected_kw': 10.0, 'actual_kw': 9.61, 'output_pct': 96.1, 'fault': False}, 'group_3': {'expected_kw': 10.0, 'actual_kw': 9.95, 'output_pct': 99.5, 'fault': False}, 'group_4': {'expected_kw': 10.0, 'actual_kw': 9.51, 

In [ ]:
print("Result keys:", result.keys())

print("\nInterrupt information:")
print(result.get("__interrupt__"))

Result keys: dict_keys(['task', 'plan', 'weather_data', 'panel_status', 'energy_forecast', 'maintenance_needed', 'retries', 'final_report', 'review_notes', 'agent_trace', 'human_approved', 'approval_message', '__interrupt__'])

Interrupt information:
[Interrupt(value={'message': 'Do you approve this report?', 'report': "Weather: {'source': 'open-meteo', 'condition': 'cloudy', 'irradiance': 0.0, 'cloud_cover_pct': 88, 'temperature_c': 42.1}\nPanels: {'group_1': {'expected_kw': 10.0, 'actual_kw': 9.82, 'output_pct': 98.2, 'fault': False}, 'group_2': {'expected_kw': 10.0, 'actual_kw': 9.61, 'output_pct': 96.1, 'fault': False}, 'group_3': {'expected_kw': 10.0, 'actual_kw': 9.95, 'output_pct': 99.5, 'fault': False}, 'group_4': {'expected_kw': 10.0, 'actual_kw': 9.51, 'output_pct': 95.1, 'fault': False}, 'group_5': {'expected_kw': 10.0, 'actual_kw': 9.51, 'output_pct': 95.1, 'fault': False}, 'group_6': {'expected_kw': 10.0, 'actual_kw': 9.77, 'output_pct': 97.7, 'fault': False}, 'group_7': {

In [ ]:
from langgraph.types import Command

resumed_result = app.invoke(
    Command(resume=True),
    config=config
)

print("Human approved:", resumed_result["human_approved"])
print("Approval message:", resumed_result["approval_message"])

Human approved: True
Approval message: Report approved by human.


In [ ]:
# ============================================================
# EVIDENCE: Retry loop under a forced low-confidence scenario
# (This is a manual test of route_after_energy's retry condition,
# not a real graph run — included to prove the loop logic fires.)
# ============================================================
test_state = {"task": "test", "plan": [], "retries": 0,
              "weather_data": {"source": "simulated_fallback", "irradiance": 50},
              "panel_status": {"group_1": {"output_pct": 40, "fault": True}}}

test_scenario = {**test_state, "energy_forecast": {"confidence": 0.3}}
decision = route_after_energy(test_scenario)
print(f"Decision with confidence=0.3 and retries=0: '{decision}'")
print(f"Retries counter after check: {test_scenario['retries']}")

Decision with confidence=0.3 and retries=0: 'retry'
Retries counter after check: 0


### Retry loop: live graph evidence

The cell above only unit-tests the routing function in isolation. The two
cells below prove the retry edge actually fires **inside a running compiled
graph** and terminates on its own condition (confidence ≥ 0.6, or 3 attempts
reached) — using a small demo graph that reuses the real `route_after_energy`
and `increment_retry_node` from the main pipeline, with a stub energy node
whose forecast confidence is deliberately low for the first two attempts and
then recovers, so the loop is guaranteed to actually run more than once.


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class RetryDemoState(TypedDict):
    energy_forecast: dict
    retries: int

def stub_energy_node(state: RetryDemoState) -> RetryDemoState:
    """Stands in for the real energy_node: returns a low-confidence forecast
    for the first two attempts, then a high-confidence one, so the retry
    loop is exercised for real instead of only unit-tested."""
    attempt = state.get("retries", 0)
    confidence = 0.3 if attempt < 2 else 0.85
    print(f"[Energy Agent - demo] attempt={attempt} -> confidence={confidence}")
    return {**state, "energy_forecast": {"confidence": confidence, "predicted_kwh": 500}}

def demo_route_after_energy(state: RetryDemoState) -> str:
    return route_after_energy({**state, "panel_status": {}})  # reuse real routing fn

retry_demo_graph = StateGraph(RetryDemoState)
retry_demo_graph.add_node("energy", stub_energy_node)
retry_demo_graph.add_node("increment_retry", increment_retry_node)  # reuse real node
retry_demo_graph.set_entry_point("energy")
retry_demo_graph.add_conditional_edges("energy", demo_route_after_energy, {
    "retry": "increment_retry",
    "continue": END,
})
retry_demo_graph.add_edge("increment_retry", "energy")

retry_demo_app = retry_demo_graph.compile()

print("=" * 60)
print("LIVE RETRY LOOP DEMO")
print("=" * 60)
retry_demo_result = retry_demo_app.invoke({"energy_forecast": {}, "retries": 0})
print("\nFinal state after the loop terminated:")
print(retry_demo_result)


LIVE RETRY LOOP DEMO
[Energy Agent - demo] attempt=0 -> confidence=0.3
[Retry Node] Confidence too low (0.30) — retrying. Attempt 1
[Energy Agent - demo] attempt=1 -> confidence=0.3
[Retry Node] Confidence too low (0.30) — retrying. Attempt 2
[Energy Agent - demo] attempt=2 -> confidence=0.85

Final state after the loop terminated:
{'energy_forecast': {'confidence': 0.85, 'predicted_kwh': 500}, 'retries': 2}
